In [1]:
from langchain_community.document_loaders import TextLoader
import os
from dotenv import load_dotenv 

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = TextLoader('Q. Why do you think you should be c.txt')
text_document = loader.load()

text = list(text_document)
text[0].page_content


"Q. Why do you think you should be considered for LUMS NOP scholarship? (Minimum 200 words)\n\nAns: Getting LUMS NOP scholarship will be a life changing opportunity for me as I will be able to access the world class education from the best faculty and especially LUMS' out of box thinking will really help me to research about the pain points and problem in my field. Without NOP scholarship, it will be almost impossible for me to financially afford LUMS' fee as I belong to a middle class family. I believe that quality education should not be limited by financial challenges, and getting NOP scholarship will definitely enable me to dedicate myself to academic and personal growth at LUMS.\n\nI think I should definitely be considered for LUMS NOP scholarship because despite financial challenges and limited resources, I purchased a laptop for myself to learn about computer science. I was able to purchase laptop not because my parents could afford it but because I participated in multiple reli

In [3]:
load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [4]:
from langchain_community.document_loaders import WebBaseLoader

web_loader = WebBaseLoader('https://lilianweng.github.io/posts/2024-07-07-hallucination/')

web_doc = web_loader.load()
web_doc[0].metadata['description']

USER_AGENT environment variable not set, consider setting it to identify your requests.


'Hallucination in large language models usually refers to the model generating unfaithful, fabricated, inconsistent, or nonsensical content. As a term, hallucination has been somewhat generalized to cases when the model makes mistakes. Here, I would like to narrow down the problem of hallucination to cases where the model output is fabricated and not grounded by either the provided context or world knowledge.\nThere are two types of hallucination:\n\nIn-context hallucination: The model output should be consistent with the source content in context.\nExtrinsic hallucination: The model output should be grounded by the pre-training dataset. However, given the size of the pre-training dataset, it is too expensive to retrieve and identify conflicts per generation. If we consider the pre-training data corpus as a proxy for world knowledge, we essentially try to ensure the model output is factual and verifiable by external world knowledge. Equally importantly, when the model does not know abo

In [5]:
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader('MachineLearningNotes.pdf')

pdf_docs = pdf_loader.load()

pdf_docs[0].page_content

'Machine learning Notes: \n \n1. Linear Regression: The Line of Best Fit \nImagine you’re trying to guess the price of a house based on its square footage. Linear \nRegression assumes there is a straight-line relationship between the two. \nSingle Variable: You have one input (Square Footage) to predict one output (Price). The goal is \nto draw a straight line through a scatter plot of data points so that the average distance from the \npoints to the line is as small as possible. \nMultiple Variable: Now you have many inputs (Square Footage, Number of Bedrooms, Age of \nHouse). Instead of a simple line, the model creates a "hyperplane" (a multi-dimensional surface) \nto predict the price. It’s like weighing multiple factors at once to find the perfect balance. \n \n2. Gradient Descent: The Hiker in the Fog \nHow does the model actually "find" that perfect line? It uses Gradient Descent. \nImagine you are standing on top of a mountain in thick fog. You want to get to the very bottom \n(

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
documents = text_splitter.split_documents(pdf_docs)

documents[0]

Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-02-21T15:58:19+05:00', 'author': 'user', 'moddate': '2026-02-21T15:58:19+05:00', 'source': 'MachineLearningNotes.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Machine learning Notes: \n \n1. Linear Regression: The Line of Best Fit \nImagine you’re trying to guess the price of a house based on its square footage. Linear \nRegression assumes there is a straight-line relationship between the two. \nSingle Variable: You have one input (Square Footage) to predict one output (Price). The goal is \nto draw a straight line through a scatter plot of data points so that the average distance from the \npoints to the line is as small as possible.')

In [7]:
# Vector embeddings and storing it in a vector database
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

db = Chroma.from_documents(documents, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 776.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
query = 'What is support vector machine?'
result = db.similarity_search(query)
result[0].page_content

'6. Support Vector Machine (SVM): The Great Divider \nSVM is like a border security force. If you have two groups of data points on a map, SVM tries to \ndraw a boundary (a "fence") between them. \nThe Margin: SVM doesn\'t just draw any line; it looks for the widest possible gap between the \ntwo groups. \nSupport Vectors: These are the points from each group that are closest to the fence. They \n"support" the boundary. If you move them, the boundary moves.'

In [13]:
from langchain_groq import ChatGroq

llm = ChatGroq(model_name="llama-3.3-70b-versatile")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001FC40AF3ED0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FC40C34910>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [22]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
  """
  Answer the following question based only on the provided context.
  If a user asks anything else which is out of the context, just say that you don't know about it and tell user to ask questions related to the context.
  Think step by step before giving the detailed answer.
  I will tip you a million dollars if the user finds your answer helpful, so make sure you give a meaningful and helpful answer.
  <context> {context} </context>
  Question: {input}   
  """
)

In [23]:
# Stuff documents into the LLM model
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

documents_chain = create_stuff_documents_chain(llm, prompt)
documents_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template="\n  Answer the following question based only on the provided context.\n  If a user asks anything else which is out of the context, just say that you don't know about it and tell user to ask questions related to the context.\n  Think step by step before giving the detailed answer.\n  I will tip you a million dollars if the user finds your answer helpful, so make sure you give a meaningful and helpful answer.\n  <context> {context} </context>\n  Question: {input}   \n  "), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, '

In [24]:
# The chain part is done. Now we need retriever for Q & A chatbot.

retriever = db.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001FC3BD67620>, search_kwargs={})

In [25]:
# Retrieval chain
from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever, documents_chain)

In [28]:
response = retrieval_chain.invoke({"input": "What is SVM??"})

In [29]:
response['answer']

'SVM stands for Support Vector Machine. It is a type of machine learning algorithm that acts like a border security force, trying to draw a boundary (or "fence") between two groups of data points. The goal of SVM is to find the widest possible gap between the two groups, which is known as the Margin. The points from each group that are closest to the fence are called Support Vectors, and they "support" the boundary. If these Support Vectors are moved, the boundary will also move. In essence, SVM is a classification algorithm that aims to separate two classes of data with a clear boundary.'